## IMPORT RELEVANT PACKAGES AND NEEDED DATA

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from tempest1d import EKF, simulate, ModelProperties, run_EKF, run_RTS

# Read in Parameter Input File and make sure
# path to VTP data (THE PATH TO YOUR INPUT DATA GOES HERE, folder paths are contained within, SAMPLE DATA IN PROPER FORMAT PROVIDED IN IOB05_obs_class.csv)
# VTP_df column header format (e.g., IB05_01cm, IB05_07cm, IB05_11cm)

input_df = pd.read_csv('Input_Example_Tempest.csv')

ModuleNotFoundError: No module named 'pandas'

In [ ]:

# path to VTP data (THE PATH TO YOUR DATA GOES HERE, SAMPLE DATA IN PROPER FORMAT PROVIDED IN IOB05_obs_class.csv)
# VTP_df column header format (e.g., IB05_01cm, IB05_07cm, IB05_11cm)
path_to_VTP_data = input_df.iloc[0,0] #'C:\\Users\\dhare\\Dropbox\\ResearchProjects\\002_USGS_DiurnalPASTA\\300_Analysis\\example_EKF\\IB1_FV01.csv'
station_str = input_df.iloc[0,1]#'IB1FV01_'  #move this up or automatically pull
path_to_save = input_df.iloc[0,2]#'C:\\Users\\dhare\\Dropbox\\ResearchProjects\\002_USGS_DiurnalPASTA\\300_Analysis\\example_EKF\\'  #'C:\\Users\\drey\\Desktop_local\\tempest\\EKF_examples\\farmington\\'
#Rey Example:
#path_to_VTP_data= 'C:\\Users\\dhare\\Dropbox\\ResearchProjects\\002_USGS_DiurnalPASTA\\300_Analysis\\example_EKF\\IB05_obs_class.csv'
#station_str= 'IB05_' #move this up or automatically pull
'''
input data required
'''

'''
define boundary condition and desired observation depths (these are in meters):
    o tBC is the top boundary condition depth
    o bBC is the bottom boundary condition depth
    o obsD is a python list of observation depths
    o measure_points is a numpy array of observation depths
    o n_measurements is the number of observation depths
'''

tBC, bBC = input_df.iloc[0,3], input_df.iloc[0,4]#'#'0.01, 0.11
obsD = [input_df.iloc[:,5]]#'[0.07]
measure_points = np.array(obsD)  # observation depths
n_measurements = len(measure_points)  # number of observation depths
# Number of seconds in a day
spd = 60. * 60. * 24.

'''
define constants for sediment and water thermal properties:
    o Kt is the thermal conductivity of sediment, W/m/C
    o Cwater is the volumetric heat capacity of water, J/m3/C
    o Csed is the volumetric heat capacity of sediment, J/m3/C
    o phi is the porosity of sediment, unitless
    o Csat is the volumetric heat capacity of saturated sediment, J/m3/C

    note: Csat can also be hardcoded if desired, below it is calculated from:
        o sediment thermal properties
        o porosity
        o water thermal properties
'''
Kt = input_df.iloc[0,6]#'2.9288
Cwater = input_df.iloc[0,7]#'4.18e6
Csed= input_df.iloc[0,8]#'2.65e6
phi= input_df.iloc[0,9]#' 0.35
Csat = input_df.iloc[0,10]#'3.5e6  #phi*Cwater + (1 - phi)*Csed # volumetric heat capacity of saturated sediment, J/m3/C,
if isinstance(Csat, numbers.Real) and not isinstance(Csat, bool) and np.isfinite(Csat):
    Csat = phi*Cwater + (1 - phi)*Csed  #convert m/day2 to m/s2 for model input
    print(Csat)
'''
1) Define range of discharge process variances
2) Print out process variance being used to calculate discharge (note process variance is in (m/s)^2, multiply by seconds per dday to get more interpretable (m/day)^2)
3) Select a process variance to use for this run
'''
#User input vq_mday2, and if not then automatic covariance is determine using...
try:
    vq_mday2 = input_df.iloc[0,11] ##float(input("Enter vq in m/day2 (press Enter for default calc):") or None)
    print(vq_mday2)
except (ValueError, TypeError):
    vq_mday2 = None

if isinstance(vq_mday2, numbers.Real) and not isinstance(vq_mday2, bool) and np.isfinite(vq_mday2):
    vq = vq_mday2 / spd / spd  #convert m/day2 to m/s2 for model input
    print("Vq is here")
else: vq_mday2 = None


if vq_mday2 is None:
    # run range of process variance
    vqs = np.logspace(-10, -20, 20)

    for ind, vq in enumerate(vqs):
        print(f"vqs[{ind}] = {vq * spd * spd} m/day2 process variance")
    vq = vqs[8]  # select a process variance to use for this run

print(f"USING vq = {vq * spd * spd} m/day2 PROCESS VARIANCE FOR EKF RUN")



## FORMAT DATA FOR EKF RUN

In [ ]:

# import VTP data .csv as a dataframe, ensure 'timestamp' column is parsed as datetime (python datetime format)
vtp_df = pd.read_csv(path_to_VTP_data, parse_dates=['timestamp'])

# Get earliest date from the timestamp column
first_date = vtp_df["timestamp"].min().date()  # assumes 'timestamp' is datetime

# Build filename like 'IB05_20200101.jpg'
fname = f"{station_str}{first_date:%Y%m%d}"



# display the dataframe to ensure it imported correctly
vtp_df



In [ ]:




'''
define model properties object:
    this sets up the finite difference model discretization and model thermal properties,
    it takes the top and bottom boundary condition depths, number of depths to discretize the model, thermal conductivity,
    volumetric heat capacity of water, and volumetric heat capacity of saturated sediment
'''

mp = ModelProperties(top_depth=tBC, bottom_depth=bBC, n_depths=41, 
                    Kt= Kt, Cw= Cwater, Cs= Csat)


'''
1) Get strings that correspond to column headers in VTP dataframe (i.e., add station string and conver from m to cm)
2) Get dataframe columns that correspond to boundary conditions and observation depths and convert to numpy arrays
'''

# string to account for VTP_df column header format (e.g., IB05_01cm, IB05_07cm, IB05_11cm)


# top boundary condition string
topStr = station_str + str(int(tBC*100)) + 'cm'

# bottom boundary condition string
botStr = station_str + str(int(bBC*100)) + 'cm'

# observation depth strings, this creates a list of strings in a recursive loop in case there are multiple observation depths, also works if only one observation depth
midStr= [station_str + str(int((entry)*100)) + 'cm' for entry in measure_points]

# Find where mp (our ModelProperties object) depths match the top and bottom boundary condition depths
i_kalman_top= np.where(mp.depths==mp.top_depth)[0][0]
i_kalman_bottom= np.where(mp.depths==mp.bottom_depth)[0][0]

# Convert columns from VTP_df to numpy arrays for top and bottom boundary conditions
T_kalman_top_all= vtp_df[topStr].to_numpy()
T_kalman_bottom_all= vtp_df[botStr].to_numpy()

# Convert columns from VTP_df to numpy arrays for observations depths, transpose so that each row is a depth and each column is a time (need to set up matrices this way for EKF matrix calculations)
measurements_all = vtp_df.loc[:, midStr].to_numpy().T

'''
1) Format datetime column as datetime format
2) Calculate time in seconds from start of observations
3) Find dt between all observations (EKF can handle gaps in time)
4) Find any NaN values in the vtp_df dataframe and filter out those times for all needed numpy arrays
5) Interpolate initial temperature state at time 0 for all depths (linear between top and bottom boundary conditions), set discharge initial guess to zero
note: EKF runs in seconds, so all time variables need to be in seconds can convert to days/hours/mintues for plotting later
'''



# Datetime columns from vtp_df
datetime_all = vtp_df['timestamp']

# Calculate time in between start and each observation in observational period
times_all = (datetime_all - datetime_all[0]).dt.total_seconds().to_numpy()

# Find any nan values in the vtp_df dataframe (note this will drop any rows where any of the columns have NaN values, so ensure that bad data columns not needed for EKF are removed from vtp_df before this step)
all_valid = ~vtp_df.isna().any(axis=1).to_numpy()

# Get only valid times in observation depths numpy arrays (i.e., obsD depths)
measurements = measurements_all[:, all_valid]

# Get only valid times in top and bottom boundary conditions numpy arrays)
T_kalman_top = T_kalman_top_all[all_valid]
T_kalman_bottom = T_kalman_bottom_all[all_valid]

# Filter datetime type numpy arrays for only valid datetime observations
datetime = datetime_all[all_valid]
times = times_all[all_valid]

# Calculate time step between all valid observations
dt = np.diff(times)

# Calculate time step between all valid observations in days
dt_days = np.diff(times/spd)

# Interpolate all measurements at time 0 to initialize state
T_initial= np.linspace(T_kalman_top[0], T_kalman_bottom[0], mp.n_depths)
q_initial = 0

# DEFINE FUNCTION TO INTIALIZE KALMAN FILTER OBJECT, SELECT PROCESS COVARIANCE VALUE

In [ ]:
'''
Create a function that initializes and returns a new EKF object with the needed properties
note: this function can be called each time a new EKF object is needed, the EKF object can be reset by calling this function again
'''
def new_EKF():
    '''
    Create a new EKF object with the needed properties:
    x0: initial state vector (internal temperatures and discharge), initial temp. guess comes from linearly interpolating between top and bottom boundary conditions, initial discharge guess is 0 m/s
    Tbc0: initial guess for boundary conditions, set to first observed temperatures in top and bottom boundary conditions
    nx: number of state variables, length of state vector, populated from x0
    n_measurements: number of observation depths, populated from measure_points
    ekf = EKF object that takes:
        o measure_points: numpy array of observation depths
        o dt[0]: time step between first two observations (EKF can handle variable time steps)
        o mp: ModelProperties object that defines the finite difference model discretization and thermal properties
        o interp=True: this tells the EKF to interpolate between model time steps
        o Tbc0=Tbc0: initial guess for temperatures
    ekf.x: initialize state variables with initial temperture and discharge guesses
    ekf.P: initial state covariance matrix (don't worry about intial parameterization) this is updated at each time step by the EKF and will be a model output.
           This represents the overall uncertainty in the Kalman filter's estimate of the system's state variables for each timestep (i.e., output at each time the state variable is estimated).
    ekf.Q: process covariance matrix, units = (m/s)^2, this is the discharge process variance, this will be changed (depending on how you set vq) for each run, just a placeholder for the funtion initialization
    T_noise_std: standard deviation of temperature observations, can populate with reported instrument error if known
    ekf.control_covariance: control covariance matrix, follows from T_noise_std (don't need to change)
    ekf.R: measurement covariance matrix, follows from T_noise_std (don't need to change)
    '''
    
    # initial state: internal temperatures concatenated with discharge, 
    x0 = np.r_[T_initial[(i_kalman_top + 1):i_kalman_bottom], q_initial]
    Tbc0 = np.r_[T_kalman_top[0], T_kalman_bottom[0]]
    nx = len(x0)
    ekf = EKF(measure_points, dt[0], mp, interp=True, Tbc0=Tbc0) 
    ekf.x = x0

    # define initial state covariance matrix
    ekf.P = np.eye(nx)*5**2
    ekf.P[-1, -1] = (1/spd)**2

    # define process covariance matric
    ekf.Q = np.eye(nx)*1e-2**2
    ekf.Q[-1, -1] = 3e-8**2

    # define measurement and control covariance matrices based on measurement error
    T_noise_std = 0.042
    ekf.control_covariance = np.eye(2)*T_noise_std**2
    ekf.R = np.eye(n_measurements)*T_noise_std**2

    return ekf



# RUN KALMAN FILTER TO GET DISCHARGE ESTIMATES

In [ ]:
'''
Run EKF and RTS smoother:
1) Call new_EKF function to get a new EKF object
2) Define process covariance matrix with selected process variance
3) Run EKF (hover cursor over run_EKF to see function argument/variable descriptions)
4) Get predicted discharge from filter as variable (m/day)
5) Get state covariance of discharge as variable
6) Compute discharge uncertainty estimates from state covariance values using standard deviation of state covariance values (P_q_ekf) to compute 95% confidence interval assuming normal distribution (e.g., std*1.96)
7) Run RTS smoother, hover cursor over run_EKF to see function argument/variable descriptions (ERTSS recursive smooths EKF output in a backwards pass using EKF forward pass output, i.e., ekf, x_ekf, P_ekf)
8) Return ERTSS predicted discahrge as a variable (m/day)
9) Calculate confidence intervals for RTS discharge estimates (from state covariance values as per EKF approach)
10) Save the outputs from filter runs to a dataframe for easy plotting and saving as .csv file
'''

# call EKF function to get a new EKF object
ekf = new_EKF()

# define process covariance matrix with selected process variance
ekf.Q[-1, -1] = vq
Qc = ekf.Q/dt[0]

# run EKF (hover cursor over run_EKF to see function argument/variable descriptions)
x_ekf, y_ekf, P_ekf = run_EKF(ekf, measurements, T_kalman_top, T_kalman_bottom,
                            dt=dt, Qc=Qc, return_full_P=True)

# get predicted discahrge from filter as variable (m/day)
q_ekf = x_ekf[:,mp.nz]*spd

# get state covariance of discharge as variable
P_q_ekf = P_ekf[:, -1, -1]

# compute discharge uncertainty estimates from state covariance values using standard deviation of state covariance values (P_q_ekf) to compute 95% confidence interval assuming normal distribution (e.g., std*1.96)
q_std = np.sqrt(P_q_ekf)*spd
q_ci = q_std*1.96

# run RTS smoother, hover cursor over run_EKF to see function argument/variable descriptions (ERTSS recursive smooths EKF output in a backwards pass using EKF forward pass output, i.e., ekf, x_ekf, P_ekf)
x_rts, y_rts, P_q_rts = run_RTS(ekf, x_ekf, P_ekf, measurements, T_kalman_top, T_kalman_bottom,
                                dt=dt, Qc=Qc, return_full_P=False)

# return ERTSS predicted discahrge as a variable (m/day)
q_rts = x_rts[:,mp.nz]*spd

# calculate confidence intervals for RTS discharge estimates (from state covariance values as per EKF approach)
q_rts_std = np.sqrt(P_q_rts)*spd
q_rts_ci = q_rts_std*1.96

# save the outputs from filter runs to a dataframe for easy plotting and saving as .csv file
'''
Save the outputs from filter runs to a dataframe for easy plotting and saving as .csv file:
o datetime: datetime of observations
o ekf_q: EKF estimated discharge (m/day)
o rts_q: RTS estimated discharge (m/day)
o ekf_T_res: EKF estimated temperature residuals at observation depths (C)
o rts_T_res: RTS estimated temperature residuals at observation depths (C)
o pcov: process variance used for the run ((m/day)^2)
note: to get estimated ekf_T and rts_T at observation depths, add the residuals to the observations
'''

filtered= pd.DataFrame({
    'datetime': datetime,
    'ekf_q': q_ekf,
    'rts_q': q_rts,
    'ekf_T_res': np.squeeze(y_ekf),
    'rts_T_res': np.squeeze(y_rts),
    'obs_T': np.squeeze(vtp_df[midStr]),
    'pcov_md2': [vq*spd*spd]*len(q_ekf),
    })

# use residuals to get estimated temperatures at observation depths
filtered['ekf_T'] = filtered['ekf_T_res'] + filtered['obs_T']
filtered['rts_T'] = filtered['rts_T_res'] + filtered['obs_T']

# add key time columns for easier plotting
filtered['day']= filtered['datetime'].dt.date
filtered['hour']= filtered['datetime'].dt.hour

# save filtered dataframe as .csv file (uncomment if you want to save the file)

full_save_path= path_to_save + 'filtered_' + fname + str(round(vq*spd*spd)) + '.csv'
print(f'filter outputs saved to {full_save_path}')
filtered.to_csv(full_save_path, index=False)
print(f'Filtered outputs example:')
filtered

## RETURN EKF AND ERTSS PREDICTED DISCHARGE SUMMARY STATISTICS

In [ ]:
ekf_stats= filtered.groupby('day')['ekf_q'].describe().reset_index()
rts_stats= filtered.groupby('day')['rts_q'].describe().reset_index()
print(f'EKF Summary Stats example:')

# uncomments to save summary daily stats for ekf and rtss runs
# ekf_stats.to_csv(path_to_save + 'ekf_stats_' + str(vq*spd*spd) + '.csv', index=False)
# rts_stats.to_csv(path_to_save + 'rts_stats_' + str(vq*spd*spd) + '.csv', index=False)

# PLOT DISCHARGE MODEL RESULTS

In [ ]:
import matplotlib.dates as mdates

# Drop the first day of data from the filtered DataFrame
first_day = filtered['day'].min()
filtered_plot = filtered[filtered['day'] != first_day]

fig,ax= plt.subplots(figsize=(30,10))
ax.plot(filtered_plot.datetime, filtered_plot.ekf_q, linewidth=2, color='xkcd:cerulean', label= 'EKF') # modeled ekf
ax.plot(filtered_plot.datetime, filtered_plot.rts_q, linewidth=2, color='xkcd:orange', label= 'RTSS', linestyle= '--') # modeled rtss

ekf_mean= np.median(ekf_stats['mean'])
rts_mean= np.median(rts_stats['mean'])
ax.annotate(f'EKF mean: {round(ekf_mean,2)}', 
             xy=(0.05, 0.10), xycoords='axes fraction', 
             color='xkcd:cerulean', fontsize=16)
ax.annotate(f'RTSS Mean: {round(rts_mean,2)}', 
             xy=(0.05, 0.05), xycoords='axes fraction',
             color='xkcd:orange', fontsize=16)

# set axis limits
# ax.set_ylim([-1.3, -0.7])

# Define label formatting
ax.set_ylabel('Specific Discharge, (m/d)', labelpad= 10, fontsize=20)

# Then set the locators
ax.minorticks_on()
ax.xaxis.set_minor_locator(mdates.AutoDateLocator())
ax.yaxis.set_minor_locator(plt.matplotlib.ticker.AutoMinorLocator())

# Make sure minor ticks are visible on all sides
ax.tick_params(which='both', top=True, bottom=True, left=True, right=True)

# Configure tick parameters for both axes
ax.tick_params(axis='both', which='major', length=5, width=1.5)
ax.tick_params(axis='both', which='minor', length=2.5, width=1)

# Set tick labels and formatting last
ax.tick_params(axis='x', labelsize=20, rotation=90)
ax.tick_params(axis='y', labelsize=20)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))

ax.legend(frameon= False, ncol= 3, fontsize=14)



fig.savefig('figures\\' + fname + '.jpeg', dpi=300)
print(f"Saved: {fname}")



## PLOT MODELED VS OBSERVED TEMPERATURES

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))

plt.savefig(fname, dpi=300, bbox_inches="tight", facecolor="white")
print(f"Saved: {fname}")
# Scatter plot for EKF
ax.scatter(filtered_plot['obs_T'], filtered_plot['ekf_T'], 
           color='xkcd:cerulean', alpha=0.5, label='EKF', s=40, edgecolor='k')

# Scatter plot for RTSS
ax.scatter(filtered_plot['obs_T'], filtered_plot['rts_T'], 
           color='xkcd:orange', alpha=0.5, label='RTSS', s=40, edgecolor='k')

# 1:1 reference line
min_val = min(filtered_plot['obs_T'].min(), filtered_plot['ekf_T'].min(), filtered_plot['rts_T'].min())
max_val = max(filtered_plot['obs_T'].max(), filtered_plot['ekf_T'].max(), filtered_plot['rts_T'].max())
ax.plot([min_val, max_val], [min_val, max_val], 'k--', linewidth=2, label='1:1 Line')

ax.set_xlabel('Observed Temperature (°C)', fontsize=18, labelpad=10)
ax.set_ylabel('Modeled Temperature (°C)', fontsize=18, labelpad=10)
ax.set_title('Observed vs Modeled Temperatures', fontsize=20, pad=15)
ax.legend(fontsize=14, frameon=False)
ax.grid(True, linestyle='--', alpha=0.5)
ax.tick_params(axis='both', labelsize=16)

plt.tight_layout()
plt.show()

fig.savefig('figures\\' + fname + 'obsVmod.jpeg', dpi=300)
print(f"Saved: {fname}")